# 04 - Collaborative Filtering

In [2]:
import sys
import os
os.chdir('/Users/mariaparis/Downloads/recommender_assignment_placeholders')
sys.path.insert(0, '/Users/mariaparis/Downloads/recommender_assignment_placeholders')

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.data_loading import load_ratings, load_items, train_test_split_ratings
from src.collaborative_filtering import ItemItemCollaborativeFiltering, UserUserCollaborativeFiltering

In [4]:
ratings = load_ratings()
items   = load_items()
train, test = train_test_split_ratings(ratings, test_size=0.2)
title_map = items.set_index(config.ITEM_COL)[config.TITLE_COL].to_dict()

## 1. Item-Item Collaborative Filtering

**How it works:** we build a user-item matrix where each row is a user and each column is a movie. Item-item similarity is computed as the cosine similarity between columns. To predict a score for a user-item pair, we look at the top-k most similar items that the user has already rated and compute a weighted average of those ratings.

**Why k=20:** using all similar items adds noise from weakly related movies. Limiting to the top 20 nearest neighbours focuses the prediction on the most reliable signals.

In [5]:
ii_cf = ItemItemCollaborativeFiltering(k=20)
ii_cf.fit(train)
print(f"User-item matrix shape : {ii_cf.user_item_matrix_.shape}")
print(f"Item similarity matrix : {ii_cf.item_similarity_.shape}")

User-item matrix shape : (610, 8983)
Item similarity matrix : (8983, 8983)


In [6]:
# Find movies most similar to Toy Story (movieId=1)
toy_story_id = 1
if toy_story_id in ii_cf.item_id_to_index_:
    idx = ii_cf.item_id_to_index_[toy_story_id]
    sims = ii_cf.item_similarity_[idx]
    top_idx = sims.argsort()[::-1][1:8]
    print("Movies most similar to Toy Story (item-item CF):")
    for i in top_idx:
        iid = ii_cf.item_ids_[i]
        print(f"  {title_map.get(iid, iid):<45} sim={sims[i]:.3f}")

Movies most similar to Toy Story (item-item CF):
  Back to the Future (1985)                     sim=0.469
  Lion King, The (1994)                         sim=0.463
  Shawshank Redemption, The (1994)              sim=0.460
  Star Wars: Episode VI - Return of the Jedi (1983) sim=0.458
  Forrest Gump (1994)                           sim=0.457
  Star Wars: Episode IV - A New Hope (1977)     sim=0.456
  Toy Story 2 (1999)                            sim=0.453


In [7]:
print("Item-Item CF recommendations for 3 users:")
for uid in [1, 50, 100]:
    recs = ii_cf.recommend(uid, train, n=5)
    print(f"\nUser {uid}:")
    for rank, iid in enumerate(recs, 1):
        print(f"  {rank}. {title_map.get(iid, iid)}")

Item-Item CF recommendations for 3 users:

User 1:
  1. Fireworks, Should We See It from the Side or the Bottom? (2017)
  2. Kizumonogatari Part 1: Tekketsu (2016)
  3. Sword Art Online The Movie: Ordinal Scale (2017)
  4. The Disaster Artist (2017)
  5. Bungo Stray Dogs: Dead Apple (2018)

User 50:
  1. Bossa Nova (2000)
  2. In Crowd, The (2000)
  3. Mouse That Roared, The (1959)
  4. 'Round Midnight (1986)
  5. Angry Red Planet, The (1959)

User 100:
  1. Sand Pebbles, The (1966)
  2. Deathgasm (2015)
  3. Guardian, The (2006)
  4. A Street Cat Named Bob (2016)
  5. General Died at Dawn, The (1936)


## 2. User-User Collaborative Filtering (extension)

**How it works:** instead of comparing items, we compare users. For a given user, we find the k most similar users (neighbours) and recommend items those neighbours liked that the target user has not seen yet.

**Item-Item vs User-User:** item-item CF is generally preferred in practice because item similarity is more stable over time than user similarity, and it scales better when there are many more users than items.

In [8]:
uu_cf = UserUserCollaborativeFiltering(k=20)
uu_cf.fit(train)
print(f"User similarity matrix : {uu_cf.user_similarity_.shape}")

User similarity matrix : (610, 610)


In [9]:
# Find users most similar to User 1
uid = 1
if uid in uu_cf.user_id_to_index_:
    u_idx = uu_cf.user_id_to_index_[uid]
    sims = uu_cf.user_similarity_[u_idx].copy()
    sims[u_idx] = 0
    top_idx = sims.argsort()[::-1][:5]
    print(f"Users most similar to User {uid}:")
    for i in top_idx:
        nb_id = uu_cf.user_ids_[i]
        print(f"  User {nb_id:<6} sim={sims[i]:.3f}")

Users most similar to User 1:
  User 266    sim=0.325
  User 368    sim=0.297
  User 45     sim=0.289
  User 480    sim=0.289
  User 217    sim=0.282


In [10]:
print("User-User CF recommendations for 3 users:")
for uid in [1, 50, 100]:
    recs = uu_cf.recommend(uid, train, n=5)
    print(f"\nUser {uid}:")
    for rank, iid in enumerate(recs, 1):
        print(f"  {rank}. {title_map.get(iid, iid)}")

User-User CF recommendations for 3 users:

User 1:
  1. Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)
  2. Godfather, The (1972)
  3. Terminator 2: Judgment Day (1991)
  4. Men in Black (a.k.a. MIB) (1997)
  5. Alien (1979)

User 50:
  1. Shawshank Redemption, The (1994)
  2. Seven (a.k.a. Se7en) (1995)
  3. Requiem for a Dream (2000)
  4. Reservoir Dogs (1992)
  5. Full Metal Jacket (1987)

User 100:
  1. Braveheart (1995)
  2. Lion King, The (1994)
  3. Schindler's List (1993)
  4. E.T. the Extra-Terrestrial (1982)
  5. Terminator 2: Judgment Day (1991)


## 3. Comparing the two CF approaches

In [11]:
uid = 1
ii_recs = set(ii_cf.recommend(uid, train, n=10))
uu_recs = set(uu_cf.recommend(uid, train, n=10))
overlap = ii_recs & uu_recs

print(f"Item-Item CF top-10 : {len(ii_recs)} items")
print(f"User-User CF top-10 : {len(uu_recs)} items")
print(f"Overlap             : {len(overlap)} items")
if overlap:
    print("Shared recommendations:")
    for iid in overlap:
        print(f"  {title_map.get(iid, iid)}")

Item-Item CF top-10 : 10 items
User-User CF top-10 : 10 items
Overlap             : 0 items


## 4. Strengths and limitations

**Strengths:**
- No item metadata needed, it works purely from user behaviour
- Can surface unexpected recommendations across genre boundaries
- User-User CF is intuitive: "people like you also enjoyed..."

**Limitations:**
- Cold-start: new users or items with no ratings cannot be handled
- Sparsity: when most users have rated few items, similarity estimates are noisy
- Scalability: computing all pairwise similarities is expensive for large catalogs